
اين نوت بوک يک pipeline کامل براي Text-to-SQL با ايده masked iterative generation (LLaDA) مي سازد.

In [ ]:
!pip uninstall -y numpy pandas pyarrow
!pip install --no-cache-dir numpy==1.26.4 pandas==2.2.2 pyarrow==14.0.1
!pip install --no-cache-dir --upgrade --force-reinstall datasets==2.21.0

!pip install -q --no-cache-dir --no-deps \
    torch==2.4.1 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install -q --no-cache-dir \
    transformers==4.45.2 \
    accelerate==0.33.0 \
    bitsandbytes==0.43.3 \
    peft==0.12.0 \
    datasets==2.21.0 \
    sqlparse tqdm 

In [7]:
import torch
import torch.nn.functional as F
import numpy as np
import re
import pandas as pd
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from torch.utils.data import DataLoader
from tqdm import tqdm




MODEL_NAME = "GSAI-ML/LLaDA-8B-Instruct"
MASK_TOKEN_ID = 126336
MAX_LEN = 512
BATCH_SIZE = 1
GRAD_ACCUM = 4
LR = 1e-4
EPOCHS = 1
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [8]:




dataset = load_dataset("gretelai/synthetic_text_to_sql", split="train")
dataset = dataset.shuffle(seed=42)


train_data = dataset.select(range(0, 1000))
val_data   = dataset.select(range(1000, 1300))
test_data  = dataset.select(range(1300, 1600))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def process_data(example):
    system_prompt = "You are a Text-to-SQL assistant. Output ONLY the SQL query."
    user_content = f"Schema:\n{example['sql_context']}\n\nQuestion:\n{example['sql_prompt']}"
    
    msgs_full = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["sql"]}
    ]
    msgs_prompt = msgs_full[:2]

    full_ids = tokenizer.apply_chat_template(msgs_full, truncation=True, max_length=MAX_LEN)
    prompt_ids = tokenizer.apply_chat_template(msgs_prompt, truncation=True, max_length=MAX_LEN, add_generation_prompt=True)
    
    return {"input_ids": full_ids, "prompt_len": len(prompt_ids)}

train_set = train_data.map(process_data).filter(lambda x: len(x["input_ids"]) > x["prompt_len"])
val_set   = val_data.map(process_data).filter(lambda x: len(x["input_ids"]) > x["prompt_len"])

def collate_fn(batch):
    input_ids = [torch.tensor(b["input_ids"]) for b in batch]
    prompt_lens = [b["prompt_len"] for b in batch]
    padded_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True, padding_value=tokenizer.pad_token_id)
    return padded_ids, torch.tensor(prompt_lens)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Filter:   0%|          | 0/300 [00:00<?, ? examples/s]

In [12]:
from accelerate import init_empty_weights, load_checkpoint_and_dispatch




quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,  
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)
model.config.use_cache = False

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()



def apply_masking(input_ids, prompt_lens):
    B, L = input_ids.shape
    noisy_ids = input_ids.clone()
    labels = torch.full_like(input_ids, -100)
    p_mask = torch.rand(B, device=DEVICE) * 0.9 + 0.05

    for i in range(B):
        maskable_mask = torch.zeros(L, dtype=torch.bool, device=DEVICE)
        
        maskable_mask[prompt_lens[i]:] = (
            (input_ids[i, prompt_lens[i]:] != tokenizer.pad_token_id) & 
            (input_ids[i, prompt_lens[i]:] != tokenizer.eos_token_id)
        )
        
        indices = torch.where(maskable_mask)[0]
        if len(indices) > 0:
            rand_vals = torch.rand(len(indices), device=DEVICE)
            to_mask_idx = indices[rand_vals < p_mask[i]]
            if len(to_mask_idx) == 0: to_mask_idx = indices[:1]
                
            noisy_ids[i, to_mask_idx] = MASK_TOKEN_ID
            labels[i, to_mask_idx] = input_ids[i, to_mask_idx]
            
    return noisy_ids, labels, p_mask

def compute_loss(batch):
    input_ids, prompt_lens = batch
    input_ids = input_ids.to(DEVICE)
    noisy, labels, p_mask = apply_masking(input_ids, prompt_lens)
    
    outputs = model(noisy, attention_mask=(noisy != tokenizer.pad_token_id))
    token_loss = F.cross_entropy(outputs.logits.view(-1, outputs.logits.size(-1)), labels.view(-1), reduction="none")
    token_loss = token_loss.view(labels.shape)
    
    
    weights = 1.0 / p_mask.view(-1, 1)
    weights = torch.clamp(weights, max=15.0) 
    
    loss = (token_loss * (labels != -100) * weights).sum() / ((labels != -100).sum() + 1e-8)
    return loss


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

trainable params: 12,582,912 || all params: 8,028,164,096 || trainable%: 0.1567


In [13]:




optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
total_steps = len(train_loader) * EPOCHS
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)

for epoch in range(EPOCHS):
    model.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]")
    for step, batch in enumerate(pbar):
        loss = compute_loss(batch)
        (loss / GRAD_ACCUM).backward()
        
        if (step + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        pbar.set_postfix(loss=loss.item())

    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="[Validation]"):
            val_loss += compute_loss(batch).item()
    print(f"Validation Loss: {val_loss / len(val_loader):.4f}")


[Validation]: 100%|██████████| 300/300 [07:11<00:00,  1.44s/it]

Validation Loss: 1.5977


In [20]:



@torch.no_grad()
def generate_sql(example, block_len=12, steps_per_block=8):
    model.eval()
    
    
    model.config.use_cache = False 

    msgs = [
        {"role": "system", "content": "You are a Text-to-SQL assistant. Output ONLY the SQL query."},
        {"role": "user", "content": f"Schema:\n{example['sql_context']}\n\nQuestion:\n{example['sql_prompt']}"}
    ]
    prompt_ids = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(DEVICE)
    
    gen_len = 96 
    seq = torch.cat([prompt_ids, torch.full((1, gen_len), MASK_TOKEN_ID, device=DEVICE)], dim=1)
    prompt_end = prompt_ids.shape[1]

    
    for start in range(prompt_end, seq.shape[1], block_len):
        end = min(start + block_len, seq.shape[1])
        
        for _ in range(steps_per_block):
            out = model(seq, attention_mask=(seq != tokenizer.pad_token_id))
            logits = out.logits[:, start:end, :]
            
            
            logits = logits / 0.7 
            
            probs = torch.softmax(logits, dim=-1)
            conf, preds = torch.max(probs, dim=-1)

            is_mask = (seq[:, start:end] == MASK_TOKEN_ID)
            if not is_mask.any(): break
            
            
            num_to_fill = max(1, int(is_mask.sum().item() * 0.3))
            
            
            conf_masked = conf.masked_fill(~is_mask, -1e9)
            _, top_idx = torch.topk(conf_masked, num_to_fill, dim=-1)
            
            for idx in top_idx[0]:
                seq[0, start + idx] = preds[0, idx]
                
    
    full_text = tokenizer.decode(seq[0, prompt_end:], skip_special_tokens=True)
    
    
    
    clean_sql = full_text.replace('assistant', '').replace('user', '').split(';')[0]
    
    
    clean_sql = re.sub(r'(\b\w+\b)(?:\s+\1)+', r'\1', clean_sql)
    
    return clean_sql.strip()

In [23]:
def normalize_sql(sql):
    if not sql: return ""
    sql = re.sub(r'[^\x00-\x7F]+', '', sql) 
    sql = sql.lower().replace('"', '').replace('`', '').replace("'", "").strip()
    if sql.endswith(';'): sql = sql[:-1]
    sql = " ".join(sql.split())
    return sql.strip()

In [22]:
print("\n>>> Starting Final Evaluation on 300 samples...")
eval_results = []

test_subset = test_data.select(range(10)) 

for i, sample in enumerate(tqdm(test_subset, desc="Evaluating")):
    pred_sql = generate_sql(sample)
    gold_sql = sample['sql']    
    n_pred = normalize_sql(pred_sql)
    n_gold = normalize_sql(gold_sql)
    is_correct = (n_pred == n_gold)    
    print(f'{i}: {is_correct}')
    eval_results.append({
        "id": i,
        "question": sample['sql_prompt'],
        "predicted": pred_sql,
        "gold": gold_sql,
        "correct": is_correct
    })

df_eval = pd.DataFrame(eval_results)
accuracy = df_eval["correct"].mean()

print(f"\n{'='*40}")
print(f"FINAL ACCURACY (Normalized EM): {accuracy:.2%}")
print(f"{'='*40}")


print("\n--- Top 5 Failures for Analysis ---")
failures = df_eval[df_eval["correct"] == False].head(5)
for _, row in failures.iterrows():
    print(f"ID: {row['id']}")
    print(f"Q: {row['question']}")
    print(f"GP: {row['gold']}")
    print(f"PP: {row['predicted']}")
    print("-" * 20)


>>> Starting Final Evaluation on 300 samples...


Evaluating:  10%|█         | 1/10 [02:09<19:26, 129.58s/it]

0: False


Evaluating:  20%|██        | 2/10 [04:14<16:53, 126.68s/it]

1: False


Evaluating:  30%|███       | 3/10 [06:21<14:48, 126.91s/it]

... (11 outputs left out)

Evaluating:  90%|█████████ | 9/10 [17:15<01:47, 107.40s/it]

8: False


Evaluating: 100%|██████████| 10/10 [19:26<00:00, 116.61s/it]

9: False

FINAL ACCURACY (Normalized EM): 0.00%

--- Top 5 Failures for Analysis ---
ID: 0
Q: What is the average unloading time (in hours) in Argentina?
GP: SELECT AVG(unloading_time_hours) FROM cargo_handling JOIN ports ON cargo_handling.port_id = ports.port_id WHERE ports.country = 'Argentina' AND cargo_handling.operation_type = 'unloading';
PP: SELECT AVG(unloading_time_hours) FROM cargo_handling WHERE operation_type = 'Argentina' AND operation_date BETWEEN '2021-01-01' AND '2021-01-02'
--------------------
ID: 1
Q: How many regulatory frameworks related to blockchain have been implemented in the Middle East and North Africa?
GP: SELECT COUNT(*) FROM frameworks WHERE country IN ('UAE', 'Saudi Arabia', 'Egypt') AND type = 'Blockchain';
PP: SELECT COUNT(*) 
FROM frameworks 
WHERE country IN ('UAE', 'Saudi Arabia', 'Egypt') AND type = 'Blockchain'
--------------------
ID: 2
Q: Insert new records into the "humanitarian_aid" table for the following disaster_id, country_name, aid_amount,

In [24]:
import re
import pandas as pd

def final_tight_normalize(sql):
    if not sql: return ""
    
    sql = re.sub(r'[^\x00-\x7F]+', '', sql)
    
    sql = sql.replace('assistant', '').replace('user', '').replace('system', '')
    
    sql = sql.lower().replace('"', '').replace('`', '').replace("'", "").strip()
    
    if sql.endswith(';'): sql = sql[:-1]
    
    sql = " ".join(sql.split())
    return sql.strip()

print(">>> Re-evaluating stored results with optimized normalization...")

new_results = []
for idx, row in df_eval.iterrows():
    n_pred = final_tight_normalize(row['predicted'])
    n_gold = final_tight_normalize(row['gold'])
    
    is_correct = (n_pred == n_gold)
    
    new_results.append({
        "id": idx,
        "is_correct": is_correct,
        "clean_pred": n_pred,
        "clean_gold": n_gold
    })


df_new = pd.DataFrame(new_results)
new_accuracy = df_new["is_correct"].mean()

print(f"\n{'='*40}")
print(f"UPDATED ACCURACY: {new_accuracy:.2%}")
print(f"{'='*40}")


print("\n--- Examples that are now CORRECT (Fixed by Normalizer) ---")

for idx, row in df_new.iterrows():
    
    if row['is_correct']:
        print(f"ID: {idx}")
        print(f"  SQL: {row['clean_pred']}")
        print(f"  Status: ✅ Match Found")
        print("-" * 20)

>>> Re-evaluating stored results with optimized normalization...

UPDATED ACCURACY: 30.00%

--- Examples that are now CORRECT (Fixed by Normalizer) ---
ID: 1
  SQL: select count(*) from frameworks where country in (uae, saudi arabia, egypt) and type = blockchain
  Status: ✅ Match Found
--------------------
ID: 4
  SQL: select avg(rating) from hotel_ratings where category = budget
  Status: ✅ Match Found
--------------------
ID: 8
  SQL: select max(ice_thickness) from ice_thickness_data where location = greenland and year = 2021
  Status: ✅ Match Found
--------------------


In [32]:
import json
with open("sql_eval_results.json", "w") as f:
    json.dump(new_results, f)

In [37]:
formatted_init = []
for item in new_results:
    formatted_init.append({
        "id": item.get('id'),
        "question": "Previous Run",
        "predicted": item.get('clean_pred'), # تبدیل نام برای همخوانی
        "gold": item.get('clean_gold'),
        "correct": item.get('is_correct')
    })

with open("sql_eval_results.json", "w", encoding='utf-8') as f:
    json.dump(formatted_init, f, ensure_ascii=False, indent=4)

# ۲. تابع اصلاح شده برای همخوانی کامل فیلدها
def run_evaluation_with_checkpoint(test_dataset, checkpoint_path="sql_eval_results.json", limit=300):
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'r', encoding='utf-8') as f:
            eval_results = json.load(f)
        print(f">>> Checkpoint found. Resuming from index {len(eval_results)}")
    else:
        eval_results = []
        print(">>> No checkpoint found. Starting from index 0")

    start_idx = len(eval_results)

    try:
        # اگر دایره تست محدود است، limit را رعایت می‌کنیم
        for i in tqdm(range(start_idx, limit), desc="Total Progress"):
            sample = test_dataset[i]
            
            pred_sql = generate_sql(sample)
            gold_sql = sample['sql']
            
            n_pred = normalize_sql(pred_sql)
            n_gold = normalize_sql(gold_sql)
            is_correct = (n_pred == n_gold)
            
            eval_results.append({
                "id": i,
                "question": sample['sql_prompt'],
                "predicted": pred_sql,
                "gold": gold_sql,
                "correct": is_correct
            })
            
            print(f"ID {i}: {is_correct}")

            if (i + 1) % 5 == 0:
                with open(checkpoint_path, 'w', encoding='utf-8') as f:
                    json.dump(eval_results, f, ensure_ascii=False, indent=4)

    except KeyboardInterrupt:
        print("\n>>> Interrupted by user. Saving progress...")
    
    finally:
        with open(checkpoint_path, 'w', encoding='utf-8') as f:
            json.dump(eval_results, f, ensure_ascii=False, indent=4)
        
        # بخش نهایی با استفاده از دیتافریم
        df_eval = pd.DataFrame(eval_results)
        
        print("\n" + "="*40)
        print(">>> Finalizing results with Tight Normalization...")
        
        final_rows = []
        for idx, row in df_eval.iterrows():
            # چک کردن وجود فیلد برای جلوگیری از Key Error
            p = row.get('predicted', '')
            g = row.get('gold', '')
            
            tn_pred = final_tight_normalize(p)
            tn_gold = final_tight_normalize(g)
            is_tight_correct = (tn_pred == tn_gold)
            
            final_rows.append({
                "id": row['id'],
                "correct": is_tight_correct,
                "clean_pred": tn_pred,
                "clean_gold": tn_gold
            })
        
        df_final = pd.DataFrame(final_rows)
        print(f"UPDATED ACCURACY: {df_final['correct'].mean():.2%}")
        print("="*40)
        
        return df_final

# اجرا
results_df = run_evaluation_with_checkpoint(test_data, limit=20)

>>> Checkpoint found. Resuming from index 10


Total Progress:  10%|█         | 1/10 [02:10<19:31, 130.13s/it]

ID 10: False


Total Progress:  20%|██        | 2/10 [04:16<17:03, 127.98s/it]

ID 11: False


Total Progress:  30%|███       | 3/10 [06:01<13:42, 117.45s/it]

... (11 outputs left out)

Total Progress:  90%|█████████ | 9/10 [17:22<01:57, 117.10s/it]

ID 18: False


Total Progress: 100%|██████████| 10/10 [18:58<00:00, 113.87s/it]

ID 19: False

>>> Finalizing results with Tight Normalization...
UPDATED ACCURACY: 30.00%
